# 01 · Test-time compute: majority voting, coverage and best-of-N

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/reasoning/01_test_time_compute.ipynb) &nbsp; 📖 Concept deck: [Sampling, voting and coverage](https://nilesh-salpe.github.io/visual-ai/reasoning/02-sampling-and-voting/) &nbsp; ⏱️ Suggested time: **35 min**

**You will build:** majority voting over sampled answers (self-consistency), a simulator of voting accuracy, and best-of-N selection with an imperfect verifier, then the gap between coverage and what a selector can actually pick.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


# A question is modelled by p, the chance that one sampled chain of thought ends in the right answer.
# Wrong samples land on one of n_wrong distinct wrong answers, uniformly.
from collections import Counter

## Task 1: Majority vote

Given a list of final answers from sampled chains of thought, return the most frequent one. Break ties by the answer that **appeared first**. (This is self-consistency: sample several reasoning paths, then marginalise over the paths by voting on the answer.)

In [ ]:
def majority_vote(answers):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_majority_vote(f):
    assert f(["12", "15", "12", "9", "12"]) == "12"
    assert f(["a", "b", "b", "a"]) == "a", "tie: the answer seen first wins"
    assert f(["7"]) == "7"

check("majority_vote", _test_majority_vote, majority_vote)

## Task 2: How accurate is voting?

Simulate `trials` questions with the same `p`. For each, draw `n` samples: correct (`"right"`) with probability `p`, otherwise one of `n_wrong` wrong answers chosen uniformly. Return the fraction of questions where the majority vote (your `majority_vote`, or the reference) is right. Use `np.random.default_rng(seed)`.

In [ ]:
def voting_accuracy(p, n, n_wrong=10, trials=2000, seed=0):
    rng = np.random.default_rng(seed)
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_voting_accuracy(f):
    one = f(0.6, 1)
    assert abs(one - 0.6) < 0.04, f"one sample is just p, got {one:.3f}"
    assert f(0.6, 15) > 0.9, "with 15 votes a 60%-likely answer almost always wins"
    assert f(0.3, 25, n_wrong=10) > 0.6, "errors scattered over many answers: the right one still wins"
    assert f(0.3, 25, n_wrong=1) < 0.1, "one dominant wrong answer: voting makes things worse"

check("voting_accuracy", _test_voting_accuracy, voting_accuracy)

## Task 3: Best-of-N with a noisy verifier

A verifier scores each sample as `correct + noise`, with `correct` in {0, 1} and noise ~ Normal(0, `noise`). Best-of-N returns the highest-scoring sample. Simulate `trials` questions with `n` samples each and return the fraction where the chosen sample is correct. With `noise = 0` this equals **coverage** (pass@n); as noise grows it falls back toward `p`.

In [ ]:
def best_of_n_accuracy(p, n, noise, trials=4000, seed=0):
    rng = np.random.default_rng(seed)
    # TODO (vectorise: arrays of shape (trials, n))
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_best_of_n_accuracy(f):
    assert abs(f(0.2, 10, 0.0) - (1 - 0.8 ** 10)) < 0.03, "a perfect verifier finds a correct sample whenever one exists"
    assert abs(f(0.2, 10, 50.0) - 0.2) < 0.03, "a useless verifier picks at random"
    assert f(0.2, 10, 0.5) > f(0.2, 10, 1.5), "a better verifier picks better"

check("best_of_n_accuracy", _test_best_of_n_accuracy, best_of_n_accuracy)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: Majority vote
def majority_vote_solution(answers):
    counts = Counter(answers)
    best = max(counts.values())
    return next(a for a in answers if counts[a] == best)

check("majority_vote (reference)", _test_majority_vote, majority_vote_solution)

In [ ]:
# @title Solution for Task 2: How accurate is voting?
def voting_accuracy_solution(p, n, n_wrong=10, trials=2000, seed=0):
    rng = np.random.default_rng(seed)
    hits = 0
    for _ in range(trials):
        ok = rng.random(n) < p
        wrong = rng.integers(0, n_wrong, n)
        answers = ["right" if o else f"w{w}" for o, w in zip(ok, wrong)]
        hits += majority_vote_solution(answers) == "right"
    return hits / trials

check("voting_accuracy (reference)", _test_voting_accuracy, voting_accuracy_solution)

In [ ]:
# @title Solution for Task 3: Best-of-N with a noisy verifier
def best_of_n_accuracy_solution(p, n, noise, trials=4000, seed=0):
    rng = np.random.default_rng(seed)
    correct = rng.random((trials, n)) < p
    score = correct + rng.normal(0, noise, (trials, n)) if noise > 0 else correct.astype(float) + rng.random((trials, n)) * 1e-9
    return float(correct[np.arange(trials), score.argmax(1)].mean())

check("best_of_n_accuracy (reference)", _test_best_of_n_accuracy, best_of_n_accuracy_solution)

---
## Experiments
One hard question type (p = 0.2): coverage keeps rising with n, but what you can actually select depends on the selector. This is the gap Large Language Monkeys reports: majority voting and reward models plateau while coverage keeps growing.

In [ ]:
p = 0.2
print("   n   coverage   majority vote   best-of-N (verifier noise 0.5)   best-of-N (noise 1.5)")
for n in [1, 4, 16, 64, 256]:
    cov = 1 - (1 - p) ** n
    print(f"{n:4} {cov:10.2f} {voting_accuracy_solution(p, n, trials=600):15.2f} {best_of_n_accuracy_solution(p, n, 0.5, trials=1500):31.2f} {best_of_n_accuracy_solution(p, n, 1.5, trials=1500):22.2f}")

---
## Takeaways
* **Coverage** (any sample right) grows with samples; **selection** (picking the right one) is the bottleneck when answers cannot be checked automatically.
* Majority voting works when errors scatter over many answers and fails when one wrong answer dominates.
* Best-of-N is only as good as its verifier: with automatic checks (unit tests, proofs) it approaches coverage, with a noisy reward model it plateaus.